# 混淆矩阵：分类结果的全景图
> 难度标签：中级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：12_模型评估与选择 | 源文件：混淆矩阵.py | 核心功能：TP/TN/FP/FN、分类报告

## 概述

混淆矩阵展示分类模型在每个类别上的预测情况。对角线是正确分类，非对角线是错误分类。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
# --- 导入库 ---
from sklearn.metrics import confusion_matrix, classification_report

# 生成数据
X, y = make_classification(
    n_samples=600, n_features=10, n_informative=5,
    n_classes=3, n_clusters_per_class=1, random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

## 数学原理

### 1. 混淆矩阵的定义

混淆矩阵 $M \in \mathbb{R}^{C \times C}$ 记录分类器的预测结果：

$$M_{ij} = |\{x : y_{\text{true}} = c_i, \; y_{\text{pred}} = c_j\}|$$

对角线元素 $M_{ii}$ 为正确分类数，非对角线 $M_{ij}$（$i \neq j$）为类别 $i$ 被误分为 $j$ 的数量。

### 2. 二分类混淆矩阵

|  | 预测正 | 预测负 |
|--|--------|--------|
| **实际正** | TP（真正） | FN（假负） |
| **实际负** | FP（假正） | TN（真负） |

### 3. 归一化混淆矩阵

按行归一化（每行除以该类别的总样本数）：

$$\hat{M}_{ij} = \frac{M_{ij}}{\sum_{k} M_{ik}}$$

$\hat{M}_{ij}$ 表示真实类别为 $c_i$ 的样本被预测为 $c_j$ 的比例。对角线值即为各类别的召回率。

### 4. 从混淆矩阵导出的指标

**整体准确率**：$\text{Acc} = \frac{\text{tr}(M)}{\sum_{ij} M_{ij}}$

**类别 $c$ 的精确率**：$P_c = \frac{M_{cc}}{\sum_i M_{ic}}$（列和归一化）

**类别 $c$ 的召回率**：$R_c = \frac{M_{cc}}{\sum_j M_{cj}}$（行和归一化）

### 5. 错误模式分析

混淆矩阵揭示模型的系统性错误：
- $M_{ij}$ 大 → 类别 $i$ 和 $j$ 容易混淆
- 某行非对角线值大 → 该类别的召回率低
- 某列非对角线值大 → 该类别的精确率低

### 6. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `confusion_matrix(y_true, y_pred)` | 返回 $M$ |
| `confusion_matrix(normalize="true")` | 行归一化 $\hat{M}$（每行和为 1） |
| `classification_report()` | 从 $M$ 导出的 P, R, F1 |
| `cm[i][j]` | $M_{ij}$：真实 $c_i$ 预测为 $c_j$ 的数量 |

### 1. 基础混淆矩阵

运行 1. 基础混淆矩阵 的代码，观察算法在该环节的行为。

In [ ]:
print("=" * 60)
print("1. 基础混淆矩阵")
print("=" * 60)

cm = confusion_matrix(y_test, y_pred)
print(f"混淆矩阵形状: {cm.shape}")
print(f"矩阵内容:\n{cm}")

# 解释混淆矩阵
n_classes = cm.shape[0]
class_names = [f"类{i}" for i in range(n_classes)]

print(f"\n{'':>10}", end="")
for j in range(n_classes):
    print(f"{'预测'+class_names[j]:>12}", end="")
print()
for i in range(n_classes):
# --- 输出结果 ---
    print(f"{'真实'+class_names[i]:>10}", end="")
    for j in range(n_classes):
        print(f"{cm[i, j]:>12}", end="")
    print()

### 2. 逐类别分析

运行 2. 逐类别分析 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("2. 逐类别分析")
print("=" * 60)

for i in range(n_classes):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp  # 其他类被预测为该类
    fn = cm[i, :].sum() - tp  # 该类被预测为其他类
    tn = cm.sum() - tp - fp - fn

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    print(f"\n{class_names[i]}:")
    print(f"  TP={tp}, FP={fp}, FN={fn}, TN={tn}")
    print(f"  精确率(Precision): {precision:.4f}")
    print(f"  召回率(Recall):    {recall:.4f}")
    print(f"  F1分数:            {f1:.4f}")

### 3. 归一化混淆矩阵

运行 3. 归一化混淆矩阵 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("3. 归一化混淆矩阵")
print("=" * 60)

# 按行归一化 (每个真实类别的预测分布)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

print("按行归一化 (每行=1.0, 展示每个真实类别的预测分布):")
print(f"{'':>10}", end="")
for j in range(n_classes):
    print(f"{'预测'+class_names[j]:>10}", end="")
print()
# --- 循环处理 ---
for i in range(n_classes):
    print(f"{'真实'+class_names[i]:>10}", end="")
    for j in range(n_classes):
        print(f"{cm_normalized[i, j]:>10.3f}", end="")
    print()

print("\n解读:")
for i in range(n_classes):
    diag = cm_normalized[i, i]
    print(f"  {class_names[i]}: {diag*100:.1f}%被正确分类", end="")
    errors = []
# --- 循环处理 ---
    for j in range(n_classes):
        if j != i and cm_normalized[i, j] > 0.01:
            errors.append(f"{class_names[j]}({cm_normalized[i, j]*100:.1f}%)")
    if errors:
        print(f", 主要误判为: {', '.join(errors)}")
# --- 继续 ---
    else:
        print()

### 4. 不同模型的混淆矩阵对比

对比不同模型或配置的性能差异。

In [ ]:
print("\n" + "=" * 60)
print("4. 不同模型的混淆矩阵对比")
print("=" * 60)

models = {
    '逻辑回归': LogisticRegression(max_iter=1000, random_state=42),
    '随机森林': RandomForestClassifier(n_estimators=100, random_state=42),
}

for name, m in models.items():
    m.fit(X_train, y_train)
    y_pred_m = m.predict(X_test)
    cm_m = confusion_matrix(y_test, y_pred_m)
    accuracy = np.trace(cm_m) / cm_m.sum()

    print(f"\n{name} (准确率={accuracy:.4f}):")
    print(f"{'':>10}", end="")
    for j in range(n_classes):
        print(f"{'预测'+class_names[j]:>10}", end="")
    print()
# --- 循环处理 ---
    for i in range(n_classes):
        print(f"{'真实'+class_names[i]:>10}", end="")
        for j in range(n_classes):
            print(f"{cm_m[i, j]:>10}", end="")
        print()

### 5. 使用classification_report

运行 5. 使用classification_report 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("5. classification_report (综合报告)")
print("=" * 60)

print(classification_report(y_test, y_pred, target_names=class_names))

### 6. 错误样本分析

运行 6. 错误样本分析 的代码，观察算法在该环节的行为。

In [ ]:
print("=" * 60)
print("6. 错误样本分析")
print("=" * 60)

misclassified = y_test != y_pred
n_errors = misclassified.sum()
n_total = len(y_test)

print(f"总样本数: {n_total}")
print(f"错误预测数: {n_errors}")
print(f"错误率: {n_errors/n_total:.4f}")

# 各类别的错误分布
print(f"\n各类别错误数:")
for i in range(n_classes):
    errors_i = np.sum((y_test == i) & misclassified)
    total_i = np.sum(y_test == i)
    print(f"  {class_names[i]}: {errors_i}/{total_i} ({errors_i/total_i*100:.1f}%)")

# 最常被混淆的类别对
print(f"\n最常被混淆的类别对:")
confused_pairs = []
for i in range(n_classes):
    for j in range(n_classes):
        if i != j:
# --- 继续 ---
            confused_pairs.append((class_names[i], class_names[j], cm[i, j]))
confused_pairs.sort(key=lambda x: -x[2])
for true_cls, pred_cls, count in confused_pairs[:5]:
    if count > 0:
        print(f"  真实={true_cls} -> 误判为{pred_cls}: {count}次")

## 关键代码解释

```python
from sklearn.metrics import confusion_matrix, classification_report
cm = confusion_matrix(y_true, y_pred)
print(classification_report(y_true, y_pred))
```

## 注意事项

1. 多分类混淆矩阵维度 = 类别数^2
2. 分类报告包含精确率、召回率、F1
3. 错误模式分析比单一指标更有价值

## 总结与延伸

以上代码展示了 **混淆矩阵** 的完整流程。

**解读要点**：
- 关注 **ROC 曲线** 下的面积（AUC）
- 观察 **交叉验证** 各折的方差
- 对比不同评估指标的适用场景

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **归一化混淆矩阵**：按行或列归一化，看比例
- **错误分析**：哪两类最容易混淆
- **代价敏感学习**：不同错误的代价不同
